In [ ]:
#r "nuget: RProvider,{{package-version}}"


# Quickstart: Using Statistical Packages

A strong R community has contributed over 20,000 packages to CRAN,
R's central package registry. The F# R Type Provider enables you to
use every single one of them from within the F# environment.

Using RRrovider, you can orchestrate R workflows and manipulate R data,
pass in F# values, and extract R values back to F#.

For this example, we simply demonstrate some basic RProvider concepts
using the built-in `stats` package.

## Example: Linear Regression

Let's perform a simple linear regression from the F# interactive,
using the R.lm function.

Once you have referenced RProvider's nuget package in your script,
library, or app, you can reference the required libraries and packages this way:



In [2]:
open RProvider
open RProvider.Operators

open RProvider.graphics
open RProvider.stats


Once the libraries and packages have been loaded,
Imagine that our true model is

Y = 5.0 + 3.0 * X1 - 2.0 * X2 + noise

Let's generate a fake dataset using F# that follows this model:



In [3]:
// Random number generator
let rng = System.Random()
let rand () = rng.NextDouble()

// Generate fake X1 and X2 
let X1s = [ for i in 0 .. 9 -> 10. * rand () ]
let X2s = [ for i in 0 .. 9 -> 5. * rand () ]

// Build Ys, following the "true" model
let Ys = [ for i in 0 .. 9 -> 5. + 3. * X1s.[i] - 2. * X2s.[i] + rand () ]


Using linear regression on this dataset, we should be able to
estimate the coefficients 5.0, 3.0 and -2.0, with some imprecision
due to the "noise" part.

Let's first put our dataset into a R dataframe; this allows us
to name our vectors, and use these names in R formulas afterwards:



In [4]:
let dataset = [ 
    "Y" => Ys
    "X1" => X1s
    "X2" => X2s ] |> R.data_frame


We can now use R to perform a linear regression.
We call the [R.lm function](http://stat.ethz.ch/R-manual/R-patched/library/stats/html/lm.html),
passing it the formula we want to estimate.
(See the [R manual on formulas](http://stat.ethz.ch/R-manual/R-patched/library/stats/html/formula.html)
for more on their somewhat esoteric construction)



In [5]:
let result = R.lm(formula = "Y~X1+X2", data = dataset)


## Extracting Results from R to F#

The result we get back from R is a R Expression.
The R Type Provider tries as much as possible to keep data
as R Expressions, rather than converting back-and-forth
between F# and R types. It limits translations
between the 2 languages, which has performance benefits,
and simplifies composing R operations. On the other hand,
we need to extract the results from the R expression
into F# types.

The [R docs for lm](http://stat.ethz.ch/R-manual/R-patched/library/stats/html/lm.html)
describes what R.lm returns: a R List. We can now retrieve each element,
accessing it by name (as defined in the documentation).
For instance, let's retrieve the coefficients and residuals,
which are both R vectors containg floats:



In [6]:
let coefficients = result?coefficients.AsVector().AsReal()
let residuals = result?residuals.AsVector().AsReal()


We can also produce summary statistics about our model,
like R^2, which measures goodness-of-fit - close to 0
indicates a very poor fit, and close to 1 a good fit.
See [R docs for the details on Summary](http://stat.ethz.ch/R-manual/R-patched/library/stats/html/summary.lm.html).



In [7]:
let summary = R.summary result

summary?``r.squared``.AsScalar()


NumericS { Sexp = { ptr = 6059992640n } }

Finally, we can directly pass results, which is a R expression,
to R.plot, to produce some fancy charts describing our model:



In [8]:
Graphics.svg 8 4 (fun _ -> R.plot result)


<?xml version='1.0' encoding='UTF-8' ?> 0.0 0.1 0.2 0.3 0.4 0.5 0.6 -1 0 1 2 Leverage Standardized residuals (function (formula, data, subset, weights, na.action, method = "qr", model ... <polyline points='81.57,-418.91 86.22,-250.46 90.86,-176.63 95.50,-132.70 100.14,-102.71 104.78,-80.53 109.42,-63.25 114.07,-49.29 118.71,-37.69 123.35,-27.84 127.99,-19.35 132.63,-11.91 137.28,-5.33 141.92,0.55 146.56,5.86 151.20,10.67 155.84,15.07 160.49,19.12 165.13,22.85 169.77,26.32 174.41,29.54 179.05,32.56 183.69,35.38 188.34,38.04 192.98,40.55 197.62,42.92 202.26,45.17 206.90,47.30 211.55,49.33 216.19,51.27 220.83,53.12 225.47,54.89 230.11,56.59 234.75,58.21 239.40,59.78 244.04,61.28 248.68,62.74 253.32,64.13 257.96,65.49 262.61,66.79 267.25,68.06 271.89,69.28 276.53,70.47 281.17,71.62 285.82,72.74 290.46,73.83 295.10,74.88 299.74,75.91 304.38,76.92 309.02,77.89 313.67,78.85 318.31,79.78 322.95,80.69 327.59,81.57 332.23,82.44 336.88,83.29 341.52,84.13 346.16,84.94 350.80,85.74 355.44,86.52 360.09,87.29 364.73,88.04 369.37,88.78 374.01,89.51 378.65,90.22 383.29,90.93 387.94,91.62 392.58,92.30 397.22,92.97 401.86,93.62 406.50,94.27 411.15,94.91 415.79,95.54 420.43,96.17 425.07,96.78 429.71,97.39 434.36,97.98 439.00,98.57 443.64,99.16 448.28,99.73 452.92,100.31 457.56,100.87 462.21,101.43 466.85,101.98 471.49,102.53 476.13,103.07 480.77,103.61 485.42,104.14 490.06,104.67 494.70,105.19 499.34,105.71 503.98,106.22 508.63,106.74 513.27,107.24 517.91,107.75 522.55,108.25 527.19,108.75 531.83,109.24 536.48,109.74 541.12,110.23 545.76,110.72 ' style='stroke-width: 0.75; stroke: #9E9E9E; stroke-dasharray: 4.00,4.00;' /><polyline points='81.57,702.06 86.22,533.61 90.86,459.78 95.50,415.86 100.14,385.86 104.78,363.68 109.42,346.41 114.07,332.44 118.71,320.84 123.35,311.00 127.99,302.50 132.63,295.07 137.28,288.48 141.92,282.60 146.56,277.30 151.20,272.48 155.84,268.08 160.49,264.04 165.13,260.30 169.77,256.84 174.41,253.61 179.05,250.60 183.69,247.77 188.34,245.11 192.98,242.61 197.62,240.24 202.26,237.99 206.90,235.85 211.55,233.82 216.19,231.89 220.83,230.04 225.47,228.27 230.11,226.57 234.75,224.94 239.40,223.38 244.04,221.87 248.68,220.42 253.32,219.02 257.96,217.67 262.61,216.36 267.25,215.10 271.89,213.87 276.53,212.69 281.17,211.53 285.82,210.42 290.46,209.33 295.10,208.27 299.74,207.24 304.38,206.24 309.02,205.26 313.67,204.31 318.31,203.38 322.95,202.47 327.59,201.58 332.23,200.71 336.88,199.86 341.52,199.03 346.16,198.21 350.80,197.42 355.44,196.63 360.09,195.86 364.73,195.11 369.37,194.37 374.01,193.64 378.65,192.93 383.29,192.23 387.94,191.54 392.58,190.86 397.22,190.19 401.86,189.53 406.50,188.88 411.15,188.24 415.79,187.61 420.43,186.99 425.07,186.37 429.71,185.77 434.36,185.17 439.00,184.58 443.64,184.00 448.28,183.42 452.92,182.85 457.56,182.28 462.21,181.73 466.85,181.17 471.49,180.63 476.13,180.08 480.77,179.55 485.42,179.02 490.06,178.49 494.70,177.96 499.34,177.45 503.98,176.93 508.63,176.42 513.27,175.91 517.91,175.41 522.55,174.90 527.19,174.41 531.83,173.91 536.48,173.42 541.12,172.93 545.76,172.44 ' style='stroke-width: 0.75; stroke: #9E9E9E; stroke-dasharray: 4.00,4.00;' /><polyline points='81.57,-651.07 86.22,-412.84 90.86,-308.44 95.50,-246.32 100.14,-203.90 104.78,-172.53 109.42,-148.10 114.07,-128.35 118.71,-111.94 123.35,-98.02 127.99,-86.01 132.63,-75.49 137.28,-66.18 141.92,-57.86 146.56,-50.36 151.20,-43.55 155.84,-37.32 160.49,-31.61 165.13,-26.32 169.77,-21.43 174.41,-16.87 179.05,-12.60 183.69,-8.61 188.34,-4.85 192.98,-1.30 197.62,2.05 202.26,5.23 206.90,8.25 211.55,11.12 216.19,13.86 220.83,16.48 225.47,18.98 230.11,21.38 234.75,23.68 239.40,25.90 244.04,28.03 248.68,30.08 253.32,32.06 257.96,33.97 262.61,35.82 267.25,37.60 271.89,39.33 276.53,41.01 281.17,42.64 285.82,44.22 290.46,45.76 295.10,47.26 299.74,48.71 304.38,50.13 309.02,51.51 313.67,52.86 318.31,54.18 322.95,55.46 327.59,56.72 332.23,57.95 336.88,59.15 341.52,60.33 346.16,61.48 350.80,62.61 355.44,63.72 360.09,64.80 364.73,65.87 369.37,66.91 374.

That's it - while simple, we hope this example illustrate
how you would go about to use any existing R statistical package.
While the details would differ, the general approach would
remain the same. Happy modelling!

